In [12]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(override=True)

True

In [2]:
llm = ChatOpenAI(model="gpt-5.4-mini")

message = "In 1 sentence, what does it mean for an AI Agent to be autonomous"

reply = llm.invoke(message)

print(reply.content)

An AI agent is autonomous if it can independently perceive its environment, decide what to do, and take actions toward its goals without needing constant human guidance.


In [3]:
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)

Silent minds that learn, choose, and weave through the digital night,  
Autonomous agents turn intention into action, swift and bright.

In [5]:
openrouter_llm = ChatOpenAI(
    model="anthropic/claude-haiku-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    max_tokens=7577
)

reply = openrouter_llm.invoke("In one sentence, what is LangChain?")
print(reply.content)

LangChain is a framework for developing applications powered by large language models that simplifies the process of chaining together LLM calls and integrating them with external data sources and tools.


In [6]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]

print(llm.invoke(messages).content)

Paris is France's capital city.


In [7]:
# the exact same call as above using plain dictionaries
messages_as_dicts = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content": "What is the capital of France?"},
]
print(llm.invoke(messages_as_dicts).content)

Paris is France’s capital city.


In [8]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

name: get_share_price
description: Return the current share price for a given ticker symbol.
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly: 241.5


In [9]:
# giving tools to the LLM model
# this returns a request to run a tool

llm_with_tools = llm.bind_tools([get_share_price])

response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

content: ''
tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_9kQ0jvmoxRRVrE9cftfUq86Q', 'type': 'tool_call'}]


In [10]:
# start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

Amazon (AMZN) is **$198.00** per share.


In [11]:
# structured output using pydantic subclass

class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

name='Amazon' ticker='AMZN' founded_year=1994
Just the ticker: AMZN


In [13]:
llm_gemini = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

message_gemini = "In 1 sentence, what does it mean for an AI Agent to be autonomous"

reply_gemini = llm_gemini.invoke(message_gemini)

print(reply_gemini.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'For an AI agent to be autonomous means it can independently perceive its environment, make decisions, and execute a sequence of actions over time to achieve a specific goal without requiring constant human intervention.', 'extras': {'signature': 'El4KXAERTTIPzY+DBX5tIOuynDB23wETZTNNoBAmP5tT9UMJgRq+Hfr3YFdtfcU/vF6IFdThtk2AH3mDNjjWX7c4Opk2VIDvxnMJ2oYVbVC9nu1N6ygivLmaeem2SkDx'}}]
